# Documentación y Bitácora de Entrenamiento: Modelo YOLO26 para Detección Vial

Este notebook recopila el historial completo de desarrollo del proyecto de detección de anomalías viales (Baches D40, Grietas D20 y Calles de Tierra) en Moreno. Sirve como registro de los experimentos realizados, los errores detectados y la evolución de la estrategia de entrenamiento.

## 1. Introducción al Proyecto

El objetivo principal es desarrollar un sistema de detección automática de desperfectos en el pavimento para la mejora de la infraestructura urbana. El proyecto utiliza la arquitectura **YOLO26**, lanzada a principios de 2026, por su capacidad superior de extracción de características y velocidad de procesamiento, y **YOLOv8**, versión más antigua, pero que sigue siendo muy utilizada por la robustez de la versión y la gran capacidad de generar resultados confiables.

## 2. Intento 1: Dataset Global y Transfer Learning con Hugging Face

### Estrategia
La primera aproximación consistió en:
1. Entrenamiento base sobre el **Dataset Global** (tal cual habíamos filtrado previamente, con alrededor de 39.000 imágenes) con un modelo de **YOLO26** en su versión **small**.
2. Fine-tuning a ese entrenamiento base utilizando el **Dataset Local Moreno V1** (descargado de Roboflow con redimensión a 640x640 y Data Augmentation preestablecido).
3. Se probó también el reentrenamiento de un modelo **YOLOv8** pre-entrenado proveniente de **Hugging Face** usando solo el **Dataset Local Moreno V1**.

### Resultados y Aprendizajes
- **Resultado:** Rendimiento extremadamente pobre (mAP50 < 0.5).
- **Diagnóstico:** Se identificó que el Dataset Global tenía un exceso masivo de imágenes de fondo (background) sin etiquetas. Esto provocaba que el modelo aprendiera que "no hay nada que detectar" la mayor parte del tiempo, arruinando la sensibilidad hacia las clases reales.

## 3. Intento 2: Dataset Mixto V1

### Estrategia
Para mitigar el problema del fondo, se creó el **Dataset Mixto V1**:
- Se filtró el Dataset Global para conservar solo imágenes con etiquetas útiles y una pequeña muestra controlada de background.
- Se mezcló con el Dataset Local Moreno V1.
- Se entrenó un nuevo modelo **YOLO26 Small (`yolo26s.pt`)**.

### Resultados y Aprendizajes
- **Resultado:** Mejora leve del mAP50 (de 0.4 a ~0.60).
- **Problemas Críticos:**
  - **Sobreajuste (Overfitting):** El modelo memorizaba los datos. La pérdida de validación subía a partir de la época 45.
  - **Pobre Precisión en Baches (D40):** El mAP50-95 era de apenas 0.27.
- **Diagnóstico Final:** 
  - **Data Augmentation Conflictivo:** El aumento estático de Roboflow se sumaba al dinámico de YOLO, forzando al modelo a memorizar patrones específicos.
  - **Resolución Insuficiente:** A 640x640, al aplicar el mosaico de YOLO, los baches pequeños perdían su detalle visual convirtiéndose en manchas indetectables.

## 4. Intento 3: Dataset Mixto V2 y Alta Resolución

### Estrategia
Esta fase representa la consolidación de los aprendizajes previos, enfocándose en la calidad del dato crudo y la capacidad del modelo para detectar objetos pequeños:

1.  **Dataset Moreno Crudo (Raw):** Se eliminó el redimensionamiento y los aumentos estáticos de Roboflow. Al usar las imágenes originales, evitamos que el modelo memorice distorsiones artificiales (overfitting prematuro) y permitimos que YOLO gestione el aumento de datos de forma dinámica durante el entrenamiento.
2.  **Inyección Global Filtrada:** Se mantuvo la estrategia de inyectar solo imágenes útiles del dataset global (RDD2022) con etiquetas `D20` y `D40`, manteniendo un control estricto sobre las imágenes de fondo (background) para no diluir la sensibilidad del modelo.
3.  **Arquitectura Medium (`yolo26m.pt`):** Escalamos de la versión *Small* a la *Medium*. Aunque es más pesada, ofrece una capacidad superior para capturar texturas asfálticas complejas y diferenciar baches de sombras o grietas finas.
4.  **Entrenamiento a 1024px:** Al duplicar la resolución estándar (de 640 a 1024), garantizamos que los baches que aparecen lejos en el horizonte mantengan suficientes píxeles para ser detectados.

> **Nota:** La creación de este nuevo `dataset_mixto_crudo`, que se utiliza para el entrenamiento de esta notebook, se encuentra desarrollado en la notebook de `04_1_documentacion_entrenamiento.ipynb`.

### Configuración Técnica y Aumentos Dinámicos
Para este intento, se personalizaron los hiperparámetros de YOLO para el dominio vial:
*   **`flipud=0.0`:** Se desactivó el giro vertical. En la conducción real, el cielo siempre está arriba y el asfalto abajo; permitir giros verticales solo confundiría al modelo.
*   **`mosaic=0.5`:** Se redujo la intensidad del mosaico (que une 4 imágenes en una) para evitar que los baches pequeños se vuelvan demasiado diminutos dentro del mosaico.
*   **`hsv_s/v=0.5`:** Se aumentó la resistencia a variaciones de saturación y brillo, simulando días nublados o sombras intensas proyectadas por árboles.

## 5. Pipeline de Entrenamiento y Configuración del Entorno

Debido al volumen de datos (10.095 imágenes en alta resolución) y al modelo que utilizamos, el entrenamiento se traslada a **Google Colab** para aprovechar sus GPUs. El código a continuación no solo ejecuta el entrenamiento, sino que gestiona la arquitectura del dataset en la nube:

1.  **Conexión con Google Drive:** Se montan las unidades para acceder de forma persistente a los datasets y guardar los pesos (`.pt`) de forma segura ante desconexiones.
2.  **Construcción del Dataset Mixto:** Se definen las rutas hacia el nuevo **Dataset Mixto V2 (Crudo)**. 
3.  **Corrección Dinámica del YAML:** Este es un paso crítico. El archivo `dataset.yaml` original suele tener rutas locales. El script lee este archivo y **reescribe dinámicamente las rutas** para que apunten a las carpetas correctas dentro de Google Drive en el entorno de Colab. Sin esto, YOLO no podría localizar las imágenes para entrenar.
4.  **Entrenamiento con Reanudación Automática:** El script final está diseñado para ser robusto. Verifica si existe un archivo `last.pt` (un entrenamiento interrumpido). Si lo encuentra, reanuda exactamente donde quedó (`resume=True`); de lo contrario, inicia el entrenamiento desde cero con los pesos pre-entrenados de YOLO26 Medium.

In [ ]:
# 1. Preparación del Entorno en Google Colab
from google.colab import drive

drive.mount("/content/drive")

# Instalación de la última versión de Ultralytics para YOLO26
%pip install -q ultralytics

In [ ]:
import os
import yaml
from ultralytics import YOLO

In [ ]:
# --- CONFIGURACIÓN DE RUTAS Y PARÁMETROS ---
ruta_mixta = "/content/drive/MyDrive/YOLO_Entrenamientos/dataset_mixto_crudo"
ruta_yaml_drive = os.path.join(ruta_mixta, "dataset.yaml")
output_project = (
    "/content/drive/MyDrive/YOLO_Entrenamientos/Training_dataset_mixto_crudo"
)
nombre_entrenamiento = "YOLO26M_Mixto_Crudo_V1"

# --- PASO 1: CORRECCIÓN DINÁMICA DEL YAML ---
# Ajustamos las rutas del dataset para que funcionen dentro de Google Drive
if os.path.exists(ruta_yaml_drive):
    with open(ruta_yaml_drive, "r") as f:
        config = yaml.safe_load(f)

    config["path"] = ruta_mixta
    config["train"] = "train/images"
    config["val"] = "val/images"
    if "test" in config:
        config["test"] = "test/images"

    with open(ruta_yaml_drive, "w") as f:
        yaml.dump(config, f)
    print(f"✅ YAML actualizado con éxito en: {ruta_yaml_drive}")
else:
    print(f"❌ ERROR: No se encontró el archivo {ruta_yaml_drive}")

In [ ]:
# --- PASO 2: ENTRENAMIENTO O REANUDACIÓN ---
# Buscamos si existe un progreso previo para no perder tiempo
ultimo_peso = os.path.join(output_project, nombre_entrenamiento, "weights", "last.pt")

if os.path.exists(ultimo_peso):
    print(f"⚠️ Entrenamiento previo detectado. Reanudando desde: {ultimo_peso}")
    model = YOLO(ultimo_peso)
    model.train(resume=True, data=ruta_yaml_drive)
else:
    print(f"🚀 Iniciando nuevo entrenamiento: {nombre_entrenamiento} (YOLO26 Medium)")
    model = YOLO("yolo26m.pt")
    model.train(
        data=ruta_yaml_drive,
        epochs=100,
        patience=25,
        batch=8,
        imgsz=1024,
        project=output_project,
        name=nombre_entrenamiento,
        save=True,
        # Hiperparámetros optimizados para baches
        hsv_s=0.5,
        hsv_v=0.5,
        degrees=5.0,
        fliplr=0.5,
        flipud=0.0,  # Desactivado para mantener orientación vial
        mosaic=0.5,  # Reducido para proteger objetos pequeños
        mixup=0.1,
    )

print("✅ Proceso finalizado.")

## 6. Resultados y Métricas Finales

Tras completar las 100 épocas de entrenamiento con el **Dataset Mixto V2 Crudo** y la arquitectura de **YOLO 26 Medium**, se obtuvieron los siguientes indicadores de rendimiento:

- El resultado global (`all`):
    - **mAP50:** 0.719
    - **Precisión:**  0.78

Sin embargo, tenemos que observar de forma más detallada los resultados obtenidos, clase por clase, que nos va a dar una mejor idea del rendimiento de este modelo:

- `D20` - Grietas:
    - **mAP50:** 0.67
    - **Precision:** 0.74
    - **Análisis:** Este es un resultado muy sólido. Las grietas son el daño más difícil de detectar porque son finas, se confunden con el alquitrán y cambian muchísimo según la luz. Que el modelo acierte casi 3 de cada 4 veces que marca una grieta (Precisión 74.6%) demuestra que la red neuronal logró extraer características complejas del asfalto.
- `D40` - Baches:
    - **mAP50:** 0.49
    - **Recall:** 0.47
    - **Análisis:** Este es el punto más débil del modelo. Un Recall del 46.9% significa que el modelo se está perdiendo más de la mitad de los baches reales que hay en la calle. Además, el mAP50 de 49% indica que le cuesta bastante trabajo.
    - **¿Por qué puede que esté pasando esto?:** Visualmente, un bache es extremadamente parecido a una mancha de humedad, una sombra de un árbol o un parche de asfalto oscuro. El modelo se volvió "conservador" y ante la duda, prefiere no marcar nada.

- `calle_tierra`:
    - **mAP50:** 0.99
    - **Recall:** 1
    - **Análisis:** Aunque a simple vista estos valores perfectos podrían hacer saltar las alarmas de un posible *overfitting* (sobreajuste), hay dos factores técnicos que explican este comportamiento. Primero, la naturaleza visual de la etiqueta: a diferencia de un bache o una grieta (que son anomalías pequeñas en un fondo), una calle de tierra ocupa la totalidad del fotograma a procesar y presenta una paleta de colores y texturas muy distinta a la de el asfalto. Para la red neuronal, es un problema de detección sumamente sencillo. Segundo, el tamaño de la muestra: contamos con apenas 31 instancias en el set de validación. Al ser un volumen estadísticamente tan bajo, el modelo no se enfrentó a suficientes "casos difíciles" (varianza) que le hicieran cometer errores. Por ende, somos conscientes de que estos números inflan artificialmente el promedio global del modelo, y por eso el rendimiento real de nuestro sistema debe evaluarse observando las métricas de D20 y D40.

## 7. Conclusión

Luego de ir cambiando de arquitectura, hiperparámetros y curando los datos iniciales, obtuvimos un modelo que representa una base sólida y se acerca a un rendimiento profesional. De todas maneras, somos conscientes de que aún falta mejorar, sobre todo en la exhaustividad (*recall*) de la etiqueta de baches (`D40`). 

Nuestro objetivo original era obtener una tasa de detección mayor en todas las clases, pero la similitud visual que existe entre un bache profundo, un parche de asfalto oscuro o una sombra, demostró ser un desafío importante para la red neuronal en esta etapa. Como resultado, el modelo tiende a ser "conservador" y ante la duda, omite la detección. 

Si bien trabajaremos para subir este *recall* a futuro, este comportamiento actual tiene un efecto colateral positivo para nuestro MVP: mantiene la tasa de falsos positivos muy baja (logrando una Precisión global del 78%). Esto significa que cuando el sistema emite una alerta, es altamente confiable, lo cual nos permite validar con éxito el flujo de datos completo y la integración con nuestra API y bases de datos sin saturar la plataforma con falsas alarmas.

# 8. Próximos pasos y trabajo a futuro

Si bien el modelo actual (`YOLO26m`) cumple con los requisitos necesarios para hacer funcionar nuestro MVP y probar la arquitectura end-to-end, el componente de Machine Learning debe seguir madurando. Para lograrlo, planteamos los siguientes pasos metodológicos:

1. **Recolección pasiva de datos locales:** A medida que las cámaras recorran el municipio, se buscará aumentar el volumen de capturas locales (Moreno) para engrosar nuestra base de datos, dándole al modelo más ejemplos reales del asfalto e iluminación de la zona.

2. **Implementación de *Human-in-the-Loop* (HITL) básico:**
   Dado que el modelo aún puede confundirse con sombras o manchas, la plataforma permitirá a los usuarios u operadores descartar fácilmente las detecciones incorrectas (falsos positivos). Estas imágenes etiquetadas como "error" se guardarán para enseñar al modelo a discriminar mejor en el futuro.

3. ***Continuous Learning* (Aprendizaje Continuo):**
   Con el nuevo volumen de imágenes recolectadas y los errores corregidos por los usuarios, no será necesario entrenar un modelo desde cero. Se realizará un *fine-tuning* (ajuste fino) sobre los pesos del modelo actual (`best.pt`), permitiendo que el sistema eleve progresivamente su tasa de detección (*recall*) de baches.

4. **Entrenamiento Mixto:** 
    Para evitar el "olvido catastrófico" (que la red neuronal olvide cómo detectar grietas al enfocarse solo en corregir baches), los futuros reentrenamientos siempre incluirán una mezcla de la nueva data local con una muestra del dataset global original.